# figs_rl

> Publication figures for trained teams and for the fixes at four agents (every training run against the warm start; one row per fix), drawn from compact data with the convention of `figs_team`.

In [ ]:
#| default_exp figs_rl

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib.pyplot as plt
import numpy as np

from explRL.figs_team import W, drawn, sem, signP, star

PANELS = (('sharePerHeld', 'clues shared / clues held'), ('soloShare', 'solves made without a shared clue'),
          ('guesses', 'guesses per board'), ('welfare', 'welfare per agent-turn'))
FIX_COLORS = dict(affordance='tab:blue', instruction='tab:orange')


@drawn
def incentives(d, paper):
    """Every trained run on the held-out boards, each against the warm start, paired on board; d: runs [label, color, {metric: {board:
    value}}] with the warm start first, refs {metric: [[x, name, linestyle]]} (paper is ignored: the figure is drawn for the web)."""
    runs, refs = d['runs'], d.get('refs', {})
    warm = runs[0][2]
    fig, axs = plt.subplots(1, len(PANELS), figsize=(16, 6.2), sharey=True)
    for ax, (m, title) in zip(axs, PANELS):
        for y, (lab, col, met) in enumerate(runs):
            v = met.get(m, {})
            if not v:
                continue
            mu, e = float(np.mean(list(v.values()))), sem(list(v.values()))
            ax.errorbar([mu], [y], xerr=[e], fmt='o', color=col, capsize=3, ms=6)
            if y:
                star(ax, mu + e, y, signP([v[s] - warm[m][s] for s in sorted(v, key=int) if s in warm[m]]), horizontal=True)
        for x, name, ls in refs.get(m, ()):
            ax.axvline(x, ls=ls, lw=1.2, color='k', alpha=0.6)
            ax.annotate(f'{name} ({x:.2f})', (x, (len(runs) - 1) / 2), xytext=(-3, 0), textcoords='offset points', rotation=90,
                        ha='right', va='center', fontsize=8, color='#566372')
        ax.set_title(title, fontsize=11)
        ax.grid(axis='x', alpha=0.3)
    axs[0].set_yticks(range(len(runs)), [r[0] for r in runs], fontsize=9)
    axs[0].invert_yaxis()
    fig.suptitle('Trained 3B teams under five reward designs, every training run shown: guessing separates the arms; '
                 'sharing and welfare vary from run to run', fontsize=12)
    fig.text(0.5, 0.005, 'Qwen2.5-3B-Instruct, 4 agents sharing one LoRA adapter, GRPO; 15 held-out boards; mean ± SEM over boards; '
             '* p < 0.05, two-sided exact sign-flip test against the warm start, paired on board; '
             'vertical lines: Claude Haiku 5.5 and rule agents on the same boards', ha='center', fontsize=8, color='#566372')
    fig.tight_layout(rect=(0, 0.03, 1, 0.95))
    return fig


@drawn
def fixes(d, paper):
    """One row per fix: the share of the gap closed (or of the damage recovered), 0 the baseline and 1 the ceiling; d: rows {label, kind,
    gap, se, p}, with kind None for a group's header. Laid out for the printed page (call with paper=True for its font sizes)."""
    fig, ax = plt.subplots(figsize=(W, 0.15 * len(d) + 0.55))
    for y, r in enumerate(d):
        if r['kind']:
            e = r['se'] if np.isfinite(r['se']) else 0.0
            ax.errorbar([r['gap']], [y], xerr=[e], fmt='o', color=FIX_COLORS[r['kind']])
            star(ax, r['gap'] + e, y, r['p'], horizontal=True, size=9)
    ax.set_yticks(range(len(d)), [r['label'] for r in d])
    for t, r in zip(ax.get_yticklabels(), d):
        if not r['kind']:
            t.set_fontweight('bold')
    for x, ls in ((0, '-'), (1, '--')):
        ax.axvline(x, color='k', lw=0.5, ls=ls)
    ax.set_ylim(len(d) - 0.5, -0.5)                      # top to bottom; the header rows have no point to set the limits
    ax.set_xlabel('gap closed or damage recovered (0 baseline, 1 ceiling)')
    for kind, c in FIX_COLORS.items():
        ax.plot([], [], 'o', color=c, label=kind)
    ax.legend(loc='upper right', frameon=False)
    fig.tight_layout()
    return fig

## Tests

In [ ]:
import tempfile
from pathlib import Path

from explRL.figs_rl import *

# a warm start and two trained runs on 15 boards; a group header row and two fixes
boards = [str(b) for b in range(900, 915)]
met = lambda base: {m: {b: base + 0.01 * i for i, b in enumerate(boards)} for m, _ in PANELS}
inc = dict(runs=[['warm start', '#555555', met(0.3)], ['individual', '#c0392b', met(0.1)], ['credit', '#2471a3', met(0.3)]],
           refs=dict(welfare=[[0.55, 'Claude Haiku 5.5', '--']]))
fx = [dict(label='Pooling', kind=None), dict(label='request board (4 agents)', kind='affordance', gap=-0.18, se=0.01, p=0.002),
      dict(label='share first (4 agents)', kind='instruction', gap=-0.14, se=float('nan'), p=0.3)]
with tempfile.TemporaryDirectory() as tmp:
    for f, d, paper in ((incentives, inc, False), (fixes, fx, True)):
        p1, p2 = Path(tmp) / f'{f.__name__}1.png', Path(tmp) / f'{f.__name__}2.png'
        f(d, p1, paper=paper), f(d, p2, paper=paper)
        assert p1.stat().st_size > 1000 and p1.read_bytes() == p2.read_bytes()
fig = incentives(inc)
assert sum(t.get_text() == '*' for a in fig.axes for t in a.texts) == 4   # individual differs from the warm start in every panel; credit in none
assert [t.get_text() for t in fig.axes[-1].texts if t.get_text() != '*'] == ['Claude Haiku 5.5 (0.55)']   # the line is labelled directly
assert fig.axes[-1].get_legend() is None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()